# Azure Databricks를 사용하여 데이터 파이프라인 설계 및 구현

> **업계 상황 — 스테이웰 호텔**>> StayWell Hotels는 유럽과 중동 전역에서 50개 이상의 호텔을 운영하고 있습니다. 매일 밤,> 자산 관리 시스템(PMS)은 예약 이벤트를 클라우드 스토리지로 내보내고 충성도를> CRM 시스템은 업데이트된 게스트 프로필을 생성합니다. 데이터 엔지니어링 팀은 다음을 구축해야 합니다.> 이 원시 데이터를 수집하고 정리 및 검증한 후 전달하는 **메달리온 파이프라인**> 수익 관리 및 고객 경험 팀이 KPI를 신뢰했습니다.>> 이 노트북의 모든 데모는 이러한 환대 시나리오를 기반으로 합니다.

In [0]:
from scripts.setup import setup, setup_10

spark.sql("USE CATALOG trainer_demo")
setup_10(spark)

## 파이프라인 작업의 설계 순서

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-order-operations-pipeline.png)

### 🎯 데모: 원시 호텔 데이터 탐색(브론즈 - 수집)
**시나리오:** StayWell Hotels의 PMS 야간 내보내기가 소스 테이블 `trainer_demo.demo_10.raw_reservations`에 도착했습니다.  
청중에게 원시 데이터를 안내하여 청소 전 브론즈 레이어의 모습을 설명합니다.

**트레이너의 요점:**  
- 이는 파이프라인의 진입점입니다. 데이터는 원본 형식으로 보존됩니다.  
- NULL, 유효하지 않은 상태, 마이너스 금액을 지적하세요. 이러한 사항은 Silver에서 처리해 드리겠습니다.  
- 증분: 매일 야간 실행 시 새 기록이 추가됩니다. 우리는 Bronze를 덮어쓰지 않습니다.

In [0]:

# ── INGEST STAGE: Peek at raw reservation data ──────────────────────────────
# Trainer: run this cell to show what the Bronze layer looks like.
# Point out NULLs, negative amounts, invalid statuses — these are intentional.

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")
print(f"Total raw reservation records: {raw.count():,}")
raw.show(10, truncate=False)


In [0]:

# ── Show data quality issues in the raw source ───────────────────────────────
# Trainer: walk through each issue to motivate the need for the Silver layer.

from pyspark.sql import functions as F

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

print("=== Data quality issues in raw_reservations ===\n")

print(f"NULL reservation_id : {raw.filter(F.col('reservation_id').isNull()).count():>6,}")
print(f"NULL guest_id        : {raw.filter(F.col('guest_id').isNull()).count():>6,}")
print(f"Negative total_amount: {raw.filter(F.col('total_amount') <= 0).count():>6,}")
print(f"Invalid status       : {raw.filter(~F.col('status').isin('confirmed','checked_in','checked_out','cancelled')).count():>6,}")
print(f"check_out < check_in : {raw.filter(F.col('check_out_date') < F.col('check_in_date')).count():>6,}")
print(f"\nTotal records        : {raw.count():>6,}")


## 노트북과 Lakeflow Pipelines를 선택하세요

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.choose-notebook-vs-lakeflow-pipelines.png)

### 🎯 데모: 노트북을 선택하는 경우와 Lakeflow Spark 선언적 파이프라인을 선택하는 경우
**시나리오:** StayWell Hotels 사용 사례에서 **절차적** 노트북 예시(단계별)와 **선언적** 파이프라인(파일 참조)을 비교하여 주요 차이점을 정렬해서 강조하세요.

**트레이너의 요점:**
- 노트북 접근 방식: 전체 Python 제어, 명시적 읽기 → 변환 → 쓰기; 복잡한 사용자 정의 논리(예: classify_cancellation_reason UDF)에 적합합니다.
- 선언적 접근 방식: 결과가 *어떤 모습*이어야 하는지 정의합니다. 프레임워크는 오케스트레이션, 증분 처리 및 재시도를 처리합니다.
- 두 접근 방식 모두 공존할 수 있습니다. 즉, 맞춤형 로직을 위한 노트북, 핵심 ETL을 위한 Lakeflow Pipelines입니다.

아래 셀은 **노트북/절차적 접근 방식**을 보여줍니다. 선언적 접근 방식은 나중에 `pipelines/10-data-pipelines/`의 파이프라인 파일을 사용하여 설명됩니다.

In [0]:

# ── PROCEDURAL APPROACH: read → transform → write in a notebook ──────────────
# Trainer: this is the notebook/imperative style.
# Contrast with the declarative @dp.table definitions in the pipeline files.

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# STEP 1: Read (explicit, imperative)
raw_reservations = spark.read.table("trainer_demo.demo_10.raw_reservations")

# STEP 2: Transform — apply business logic in Python
#  Custom logic: classify booking channel into a broader segment.
#  This kind of conditional Python logic is difficult to express declaratively.
def classify_channel(channel):
    if channel in ("direct",):           return "direct"
    if channel in ("ota",):              return "online_intermediary"
    if channel in ("corporate",):        return "b2b"
    if channel in ("travel_agent",):     return "b2b"
    return "other"

classify_udf = F.udf(classify_channel, StringType())

transformed = (
    raw_reservations
    .filter(F.col("reservation_id").isNotNull())
    .filter(F.col("total_amount") > 0)
    .withColumn("channel_segment", classify_udf(F.col("booking_channel")))
    .withColumn(
        "stay_nights",
        F.datediff(F.col("check_out_date"), F.col("check_in_date"))
    )
)

# STEP 3: Write (explicit, imperative)
(
    transformed
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("trainer_demo.demo_10.notebook_reservations_enriched")
)

print(f"Written {transformed.count():,} rows to notebook_reservations_enriched")
print("\nSample output:")
transformed.select("reservation_id", "property_id", "booking_channel",
                   "channel_segment", "stay_nights", "total_amount").show(8)


## Lakeflow 작업 로직 설계

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-task-logic-lakeflow-job.png)

### 🎯 데모: Lakeflow Job — 작업 논리, DAG 종속성 및 작업 값
**시나리오:** StayWell의 야간 파이프라인은 다음 네 가지 작업이 포함된 Lakeflow 작업으로 조정됩니다.  
1. `ingest_reservations` — 브론즈 스트리밍 수집 노트북 실행  
2. `clean_reservations` — 은 세척  
3. `clean_guests` — 실버 손님 청소 (작업 2와 **병렬** 실행)  
4. `build_gold_kpis` — 작업 2와 3(팬인)에 따라 다릅니다.  

**트레이너의 요점:**  
- Lakeflow Jobs UI를 열고 DAG를 실시간으로 그려서 병렬 작업과 순차 작업을 지적합니다.  
- 학습 단위의 **작업 종속성 조건** 테이블을 표시합니다 (모두 성공 / 실패 없음 / 모두 완료).  
- 아래 셀은 작업이 테이블에 쓰지 않고 다운스트림 작업에 데이터를 전달하는 방법인 `dbutils.jobs.taskValues`을 보여줍니다.  
- `for_each` 지역에서는 10개의 호텔 속성을 각각 독립적으로 처리한다는 점을 지적하세요.

In [0]:

# ── Demonstrate dbutils.jobs.taskValues ──────────────────────────────────────
# Trainer: explain that in a real Lakeflow Job each task would be a separate
# notebook. Here we simulate setting and getting task values in-notebook for demo.

# Simulate Task 1 (ingest_reservations) setting a task value
# In a real job this code would live in the ingest_reservations notebook.

from pyspark.sql import functions as F

reservations = spark.read.table("trainer_demo.demo_10.raw_reservations")
record_count = reservations.count()

# In a Lakeflow Job this would be:
#   dbutils.jobs.taskValues.set(key="reservations_ingested", value=record_count)
print(f"[Simulated task value] reservations_ingested = {record_count:,}")

# Demonstrate getting property list for a For each task
properties = (
    reservations
    .select("property_id")
    .distinct()
    .orderBy("property_id")
    .rdd.flatMap(lambda r: [r[0]])
    .collect()
)

# In a real job:
#   dbutils.jobs.taskValues.set(key="properties_to_process", value=properties)
print(f"\n[Simulated task value] properties_to_process = {properties}")
print(f"\nA 'For each' task would iterate over these {len(properties)} properties.")
print("Each iteration processes one property independently (configurable concurrency).")


In [0]:

# ── Demonstrate serial vs parallel execution in the pipeline DAG ─────────────
# Trainer: show that tasks without dependencies run in parallel.
# Compare time for serial processing (one property at a time) vs parallel.

import time
from pyspark.sql import functions as F

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

# SERIAL: process each property one after another
print("=== Serial processing (sequential, one property at a time) ===")
t0 = time.time()
for prop in properties[:3]:  # show just 3 properties for demo speed
    count = raw.filter(F.col("property_id") == prop).count()
    print(f"  {prop}: {count:,} reservations")
serial_time = time.time() - t0
print(f"  Elapsed (serial, 3 properties): {serial_time:.1f}s\n")

# PARALLEL (conceptual): in a real Lakeflow Job these would run as parallel tasks
# The framework handles scheduling and resource allocation automatically.
print("=== Parallel processing (as Lakeflow For each task with concurrency=3) ===")
print("  In a Lakeflow Job, all 3 properties would run concurrently.")
print("  Elapsed would be ~1/3 of serial time (limited by slowest partition).")
print("\n  Configure concurrency on the 'For each' task in the job UI.")


## 파이프라인 및 작업의 설계 오류 처리

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-error-handling-pipelines.png)

### 🎯 데모: 노트북의 오류 처리 — try/exc, 재시도 및 dbutils.notebook.exit
**시나리오:** StayWell의 취소 분석 노트북은 일시적인 오류를 적절하게 처리하고 그 결과를 다시 Lakeflow 작업에 신호로 보내야 합니다.

**트레이너의 요점:**
- 인프라(Azure Databricks에서 처리), 데이터 수준, 애플리케이션 논리의 세 가지 오류 처리 수준을 정렬해서 표시합니다.
- `try/except`을 통과하여 `getErrorClass()`로 `PySparkException`을 잡으세요.
- 일시적인 오류(예: PMS API에서 읽는 네트워크 시간 초과)의 경우 `run_with_retry()`을 표시합니다.
- `dbutils.notebook.exit()` 표시 - 다운스트림 작업이 조건부 종속성의 종료 값을 사용합니다.
- 선언적 파이프라인 대안(예상)이 파이프라인 파일에 정렬해서 표시됩니다.

In [0]:

# ── Error handling: try/except with PySparkException ─────────────────────────
# Trainer: demonstrate data-level error handling in a notebook pipeline task.
# This is error handling the data engineer is responsible for (NOT infrastructure).

from pyspark.errors import PySparkException
from pyspark.sql import functions as F

# Simulate handling a missing table gracefully
try:
    # Attempt to read the silver reservations (may not exist if pipeline hasn't run)
    silver = spark.read.table("trainer_demo.demo_10.silver_reservations")
    count = silver.count()
    print(f"✓ silver_reservations has {count:,} records")

except PySparkException as e:
    if e.getErrorClass() == "TABLE_OR_VIEW_NOT_FOUND":
        print(f"⚠ Table not found: {e.getMessageParameters().get('relationName', 'unknown')}")
        print("  Downstream steps will be skipped. Run the Lakeflow Pipeline first.")
    else:
        raise  # Re-raise unexpected errors

except Exception as e:
    print(f"✗ Unexpected error: {e}")
    raise


In [0]:

# ── Error handling: retry with exponential backoff ───────────────────────────
# Trainer: show how to handle transient failures (e.g., network call to PMS API).
# StayWell's PMS API occasionally returns 503 during peak booking windows.

import time

def run_with_retry(operation, max_retries=3, base_delay=2):
    """Retry an operation with exponential backoff."""
    for attempt in range(max_retries + 1):
        try:
            return operation()
        except Exception as e:
            if attempt == max_retries:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)

# Example: simulate a flaky read operation
import random

attempt_counter = {"n": 0}

def flaky_reservation_count():
    """Simulate an operation that fails on first 2 attempts."""
    attempt_counter["n"] += 1
    if attempt_counter["n"] < 3:
        raise ConnectionError("PMS API timeout — transient error")
    return spark.read.table("trainer_demo.demo_10.raw_reservations").count()

print("Calling flaky_reservation_count() with retry logic...")
count = run_with_retry(flaky_reservation_count, max_retries=3, base_delay=1)
print(f"\n✓ Successfully retrieved count after {attempt_counter['n']} attempt(s): {count:,} records")


## 노트북으로 파이프라인 생성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.create-pipeline-notebook-precedence.png)

### 🎯 데모: 노트북 작업 및 작업 종속성이 포함된 파이프라인 생성
**시나리오:** StayWell의 취소 분석 노트북을 작업으로 실행하는 Lakeflow 작업을 구축합니다.

**트레이너를 위한 단계별 지침:**
1. Databricks 작업 영역에서 **작업 및 파이프라인** → **만들기** → **작업**을 엽니다.
2. 작업 이름을 `staywell_nightly_pipeline`로 지정합니다.
3. **작업 1**을 추가합니다.  
   - 유형 = 노트북  
   - 이름 = `ingest_reservations`  
   - 경로 = `Demo/pipelines/10-data-pipelines/04_notebook_pipeline_task`
4. **작업 2**를 추가합니다.  
   - 유형 = 노트북  
   - 이름 = `build_cancellation_report`  
   - 종속성: `ingest_reservations`에 따라 다름  
   - 매개변수 `process_date`를 `{{job.start_time.iso_date}}`로 설정
5. DAG를 표시합니다.  
   - 작업 2는 작업 1이 성공한 후에만 실행됨을 강조하세요.
6. **지금 실행**을 클릭하고 → 작업 그래프가 순차적으로 실행되는 것을 확인하세요.

**강조할 핵심 개념:**  
- 작업 종속성  
- 위젯을 통한 매개변수 전달  
- `dbutils.notebook.exit()` 신호  
- 조건부 작업 흐름

작업 1의 코드는 `pipelines/10-data-pipelines/04_notebook_pipeline_task.py`에 있습니다.

In [0]:

# ── Notebook task: widgets and parameter passing ─────────────────────────────
# Trainer: paste this into the FIRST CELL of the notebook task you create in
# the Lakeflow Job. It shows how parameters flow from the job into the notebook.
#
# When run interactively (in this demo notebook), the widget defaults are used.
# When run as a job task the job can override the values via the task parameters.

dbutils.widgets.text("process_date",  "", "Process Date (YYYY-MM-DD)")
dbutils.widgets.text("property_id",   "ALL", "Property ID (or ALL)")

process_date = dbutils.widgets.get("process_date") or str(spark.sql("SELECT current_date()").collect()[0][0])
property_id  = dbutils.widgets.get("property_id")

print(f"Running cancellation analysis for date: {process_date}, property: {property_id}")


In [0]:

# ── Notebook task: dbutils.notebook.exit to signal result to job ─────────────
# Trainer: this cell shows how a notebook communicates success/failure back
# to the Lakeflow Job. The exit value can be read by downstream tasks.
#
# The downstream task can use an If/else condition:
#   {{tasks.ingest_reservations.values.cancellations_processed}} > 0
# to decide whether to send a notification or skip reporting.

from pyspark.sql import functions as F

try:
    raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

    cancelled = raw.filter(F.col("status") == "cancelled")
    if property_id != "ALL":
        cancelled = cancelled.filter(F.col("property_id") == property_id)

    records = cancelled.count()

    # Set a task value so downstream tasks can read it
    # dbutils.jobs.taskValues.set(key="cancellations_processed", value=records)
    print(f"✓ Found {records:,} cancelled reservations")
    print(f"  (In a Lakeflow Job: dbutils.notebook.exit('SUCCESS: {records}'))")

except Exception as e:
    print(f"✗ Error: {e}")
    # In a Lakeflow Job: dbutils.notebook.exit(f"FAILED: {str(e)}")
    # The job would mark this task as failed and trigger any configured alerts.


## Lakeflow Spark 선언적 파이프라인을 사용하여 파이프라인 만들기

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.create-pipeline-lakeflow-declarative.png)

### 🎯 데모: StayWell 호텔을 위한 Lakeflow Spark 선언적 파이프라인 생성
**트레이너를 위한 단계별 지침:**
1. Databricks 작업 영역에서 **작업 및 파이프라인** → **만들기** → **ETL 파이프라인**을 엽니다.
2. 파이프라인 이름을 `staywell_hotel_pipeline`로 지정합니다.
3. **소스 코드**를 `Demo/pipelines/10-data-pipelines` 폴더로 설정합니다(또는 개별 파일을 추가합니다).
4. **대상 스키마**를 `trainer_demo.demo_10`로 설정합니다.
5. **파이프라인 모드**를 **트리거됨**(일괄 데모의 경우) 또는 **연속**(스트리밍의 경우)으로 설정합니다.
6. **저장** → **시작**(또는 실행하지 않고 확인하려면 **드라이 런**)을 클릭합니다.

**파이프라인 파일을 순서대로 정렬해서 살펴보세요.**
| File                        | Layer  | What it demonstrates                                 |
|-----------------------------|--------|------------------------------------------------------|
| `01_bronze_reservations.py` | Bronze | `@dp.table` for streaming ingestion, append-only     |
| `02_silver_clean_validate.py`| Silver | Expectations: warn / drop / fail                     |
| `03_gold_transforms.py`     | Gold   | `@dp.materialized_view` for aggregations and joins   |

**러닝 후 강조할 개념:**
- **DAG 보기** — 자동 종속성 순서 지정으로 오케스트레이션 코드를 작성할 필요가 없습니다.
- **데이터 품질 탭** — 각 기대치에 대한 합격/실패 수를 계산합니다.
- **증분 처리**: 각 실행마다 새로운 브론즈 기록만 처리됩니다.
- `@dp.table`(스트리밍, 추가 전용)과 `@dp.materialized_view`(재계산 가능, 업데이트 처리)을 대조합니다.

In [0]:

# ── Preview the declarative pipeline source files ────────────────────────────
# Trainer: read and display the pipeline source files so the audience can see
# the declarative syntax alongside the running pipeline in the UI.

import os

pipeline_dir = "pipelines/10-data-pipelines"
pipeline_files = sorted([
    f for f in os.listdir(pipeline_dir) if f.endswith(".py")
])

print(f"Pipeline source files in {pipeline_dir}/:\n")
for fname in pipeline_files:
    fpath = os.path.join(pipeline_dir, fname)
    with open(fpath) as fh:
        first_lines = [l.rstrip() for l in fh.readlines()[:8]]
    print(f"{'─'*60}")
    print(f"  📄 {fname}")
    print(f"{'─'*60}")
    for line in first_lines:
        print(f"  {line}")
    print()


In [0]:

# ── After running the pipeline: query the Gold layer results ─────────────────
# Trainer: run this AFTER the Lakeflow pipeline completes at least one run.
# Show the audience the final Gold tables that the BI tools consume.

from pyspark.sql import functions as F

try:
    # Revenue summary by property
    print("=== gold_property_revenue_summary ===")
    spark.read.table("trainer_demo.demo_10.gold_property_revenue_summary") \
         .orderBy(F.col("monthly_revenue").desc()) \
         .show(10)

except Exception as e:
    print(f"⚠ Gold tables not available yet — run the Lakeflow Pipeline first.\n  Error: {e}")

try:
    # Top 5 properties by daily occupancy
    print("=== gold_daily_occupancy (top 5 days by revenue) ===")
    spark.read.table("trainer_demo.demo_10.gold_daily_occupancy") \
         .orderBy(F.col("total_revenue").desc()) \
         .show(5)

except Exception as e:
    print(f"⚠ gold_daily_occupancy not available — run the Lakeflow Pipeline first.\n  Error: {e}")
